# MedScope 02: baselines

Runs two open models zero-shot on the MedScope-Bench test views with the agent prompt:
`Qwen/Qwen3.5-4B` (our base model) and `google/medgemma-1.5-4b-it` (Google's medical model).

**Runtime:** L4 GPU (A100 is faster). Roughly 1-2 hours per model. Results resume if Colab disconnects:
just run all cells again.

**Before you start:** in the Colab sidebar, open the key icon (Secrets) and add:
- `HF_TOKEN`: a Hugging Face read token. For MedGemma, first accept its terms at https://huggingface.co/google/medgemma-1.5-4b-it
- `GITHUB_TOKEN`: only needed if the repo is private.

Turn on notebook access for both secrets when Colab asks.

In [ ]:
# --- setup: same in every notebook ---
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json
DRIVE = '/content/drive/MyDrive/medscope'
WORK = '/content/work'
RESULTS = f'{DRIVE}/results'
ADAPTERS = f'{DRIVE}/adapters'
VIEWS = f'{WORK}/views'
for d in (DRIVE, RESULTS, ADAPTERS, WORK):
    os.makedirs(d, exist_ok=True)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

gh = secret('GITHUB_TOKEN')
url = f'https://{gh}@github.com/bhaskargurram-ai/medscope' if gh else 'https://github.com/bhaskargurram-ai/medscope'
if os.path.exists('/content/medscope/.git'):
    !git -C /content/medscope pull -q
else:
    !git clone -q {url} /content/medscope
%cd /content/medscope
!git log --oneline -1
!pip install -q -e .

In [ ]:
# --- model libraries and Hugging Face login ---
!pip install -q -U transformers accelerate peft bitsandbytes
hf = secret('HF_TOKEN')
if hf:
    os.environ['HF_TOKEN'] = hf
    from huggingface_hub import login
    login(hf)
else:
    print('No HF_TOKEN secret: MedGemma (gated) will fail to download; Qwen works without it.')
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: Runtime > Change runtime type > L4 or A100')

In [ ]:
# --- restore the views built in notebook 01 (paths inside the manifests point to /content/work/views) ---
if not os.path.exists(f'{VIEWS}/test/views_clean.jsonl'):
    !tar -C {WORK} -xf {DRIVE}/slake/views.tar
!wc -l {VIEWS}/*/views_clean.jsonl

In [ ]:
LIMIT = ''   # e.g. '--limit-cases 50' for a quick first look; '' runs every test case

## Qwen3.5-4B, zero-shot

In [ ]:
!python scripts/evaluate.py --model Qwen/Qwen3.5-4B --views {VIEWS}/test/views_clean.jsonl --out {RESULTS}/base_qwen35_test --name base-qwen3.5-4b --split test {LIMIT}

## MedGemma 1.5 4B, zero-shot

In [ ]:
!python scripts/evaluate.py --model google/medgemma-1.5-4b-it --views {VIEWS}/test/views_clean.jsonl --out {RESULTS}/base_medgemma15_test --name base-medgemma-1.5-4b --split test {LIMIT}

## Publish the report

In [ ]:
from medscope.colab import build_report, publish_doc, read_text, train_log_summary
from medscope.eval.runner import compare_runs
report = build_report('MedScope report - 02 baselines', [

    ('Leaderboard', compare_runs(RESULTS)),
    ('Qwen3.5-4B details', read_text(f'{RESULTS}/base_qwen35_test/report.md')),
    ('MedGemma 1.5 4B details', read_text(f'{RESULTS}/base_medgemma15_test/report.md')),

])
open(f'{RESULTS}/02_baselines_report.md', 'w').write(report)
publish_doc('MedScope report - 02 baselines', report)
print(report[:3000])